[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/cga_spherical_quadrics/cga_spherical_quadrics.ipynb)

# Spherical Quadrics in a Conformal Flow

This example constructs curved regions on the unit sphere and carries them around the sphere by a smooth conformal flow that preserves angles while stretching and shrinking lengths. Each region is modelled as a spherical quadric, defined implicitly as the set of points where a quadratic form on the sphere is negative. Two off-axis circles intersect at a pair of fixed points, and a full turn of the flow around those points returns every deformed region to its starting shape.

In the conformal algebra of the two-sphere ([Dorst, Fontijne and Mann, 2007](https://geometricalgebra.org/downloads/ga4cs_chapter1.pdf)), a circle cut out by a plane is a vector `Plane`, and a point of the sphere is modelled as a null antivector `Point` formed by adding a unit spatial direction to the sphere's centre `w.dual()`, so `circle & point` is positive inside the circle. A spherical quadric is a polarity map `Plane <- Point` built as a sum of symmetric circle dyads such as `outer * (inner & Point) + inner * (outer & Point)`, with interior `quadric(point) & point < 0`. Two intersecting circles meet in the 2-blade `first_circle ^ second_circle`, whose exponential `rotor` carries any quadric by pulling input points back and pushing output planes forward as `rotor >> quadric(rotor << Point)`.

In matrix Möbius geometry on homogeneous Lorentzian coordinates $X = (x, y, z, 1)^\top \in \mathbb{R}^{3,1}$ with $\eta = \operatorname{diag}(1, 1, 1, -1)$, a spherical quadric is represented by a symmetric $4 \times 4$ matrix $Q$ evaluated via $X^\top Q X < 0$, two circles with normal-offset vectors $c_1, c_2$ generate a skew-adjoint boost-rotation generator $A = \eta (c_1 c_2^\top - c_2 c_1^\top)$, and carrying $Q$ along the flow requires the inverse-congruence matrix update $Q(\theta) = \exp(-\theta A)^\top Q \exp(-\theta A)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import show_states
from examples.quadrics.cga_spherical_quadrics import render

np.set_printoptions(precision=4, suppress=True)

ga = Algebra("x+y+z+w-")
mv = NumpyContext(ga).multivector
# A circle on the unit sphere is the plane that cuts it out.
Plane = ga.gatype.vector()                                                     # Vector+
# A point of the unit sphere is a null antivector with intrinsic orientation.
Point = ga.gatype.antivector()                                                 # Antivector-
Quadric = ga.gatype((Plane, Point))                                            # Plane+ <- Point-
Bivector = ga.gatype.bivector()                                                # Bivector+

## Spherical quadrics from circle dyads

A circle on the sphere pairs positively with points inside its cap and negatively outside, so the symmetric dyad of two circles is negative wherever the two pairings have opposite signs. Combining circle dyads produces rings that pinch to a conical apex at the pole, figure-eight lemniscates, and two-focus Cassini hourglasses.

In [ ]:
# Shape parameters for the conical donut, Bernoulli lemniscate, and Cassini hourglass, with their display colours.
donut_tilt, donut_width, donut_view = 0.55, 0.22, -0.20
lemniscate_spread = 0.38
hourglass_tilt, hourglass_waist = np.radians(24), 0.010
colors = ["#38bdf8", "#a855f7", "#f59e0b"]

In [ ]:
def conical_donut(tilt: float, width: float) -> Quadric:
    """A tilted circle through the north pole minus weighted coordinate-plane dyads defines a limaçon ring quadric
    (`[] Plane+ <- Point-`) whose central hole closes in a conical apex at the pole."""
    # A circle through the north pole, tilted toward x.
    through_pole = mv.w + mv.z + tilt * mv.x                                   # [] Plane+
    return through_pole * (through_pole & Point) - width**2 * (mv.x * (mv.x & Point) + mv.y * (mv.y & Point))


def bernoulli_lemniscate(spread: float) -> Quadric:
    """The point circle at the north pole minus a hyperbolic pair of coordinate-plane dyads defines a figure-eight
    lemniscate quadric (`[] Plane+ <- Point-`) of angular spread `spread`."""
    pole = mv.w + mv.z                                                         # [] Plane+
    return pole * (pole & Point) - 2 * spread**2 * (mv.x * (mv.x & Point) - mv.y * (mv.y & Point))


# Tilt the conical donut into view by pulling input points back and pushing output planes forward.
view_tilt = (mv.xz * donut_view).exp()                                         # [] Rotor+
donut = view_tilt >> conical_donut(donut_tilt, donut_width)(view_tilt << Point)  # [] Plane+ <- Point-
lemniscate = bernoulli_lemniscate(lemniscate_spread)                           # [] Plane+ <- Point-

In [ ]:
def spherical_hourglass(tilt: float, waist: float) -> Quadric:
    """Two tangent circles tilted by `tilt` and `-tilt` along `y` define a symmetric hourglass Cassini quadric
    (`[] Plane+ <- Point-`) with waist parameter `waist`."""
    # Turning z either way in the yz plane places the two foci symmetrically above and below the pole.
    first_turn = (mv.zy * (-tilt / 2)).exp()                                   # [] Rotor+
    second_turn = (mv.zy * (tilt / 2)).exp()                                  # [] Rotor+
    first_tangent = mv.w + (first_turn >> mv.z)                                # [] Plane+
    second_tangent = mv.w + (second_turn >> mv.z)                              # [] Plane+
    return (0.5 * (first_tangent * (second_tangent & Point) + second_tangent * (first_tangent & Point))
            - waist * mv.w * (mv.w & Point))


hourglass = spherical_hourglass(hourglass_tilt, hourglass_waist)               # [] Plane+ <- Point-
quadrics = stack([donut, lemniscate, hourglass])                               # [3] Plane+ <- Point-
print("lemniscate polarity map:\n", lemniscate.coefficient_table())

## Conformal vortex from two intersecting circles

Two circles tilted away from the pole intersect in two points on the sphere, so their wedge `first_circle ^ second_circle` has negative square and exponentiates to a periodic one-parameter family of conformal rotors. Streaming those rotors over one turn carries every quadric around the two fixed points simultaneously.

In [ ]:
# Tilts and angular radii of the two circles generating the flow, and the animation frame count and resolution.
first_tilt, first_radius = 0.35, np.radians(48)
second_tilt, second_radius = 0.40, np.radians(52)
n_frames, resolution, frame_duration_ms = 50, 300, 40

In [ ]:
def conformal_flow(quadrics: Quadric, generator: Bivector, n_frames: int) -> Iterator[Quadric]:
    """Yields the `[n_quadrics]` quadrics (`[n_quadrics] Plane+ <- Point-`) across `n_frames` uniformly spaced phases
    of one full conformal turn generated by `generator`."""
    for phase in np.linspace(0, 2 * np.pi, n_frames, endpoint=False):
        rotor = (generator * (phase / 2)).exp()                                # [] Rotor+
        yield rotor >> quadrics(rotor << Point)


# Each circle is a tilted plane shifted by the cosine of its angular radius along w.
first_turn = (mv.zx * (-first_tilt / 2)).exp()                                 # [] Rotor+
second_turn = (mv.zy * (-second_tilt / 2)).exp()                               # [] Rotor+
first_circle = (first_turn >> mv.z) + np.cos(first_radius) * mv.w              # [] Plane+
second_circle = (second_turn >> mv.z) + np.cos(second_radius) * mv.w           # [] Plane+
generator = (first_circle ^ second_circle).normalized()                        # [] Bivector+

states = show_states(
    conformal_flow(quadrics, generator, n_frames),
    render.vortex_animation,
    colors,
    resolution,
    name="cga_trio_vortex",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

On the front hemisphere, the blue conical donut, purple figure-eight lemniscate, and amber Cassini hourglass stretch and compress as they orbit the two fixed intersection points while preserving the cusp and node angles of each boundary.

In [ ]:
# checks
# Every streamed quadric stays self-adjugate under polarity pullback, and one more step closes the periodic turn.
def sphere_points(xyz: np.ndarray) -> Point:
    """Unit Cartesian coordinates of shape `[..., 3]` lift to null conformal points (`[...] Point-`) on the unit sphere
    by adding the sphere's centre to their spatial directions."""
    return mv.w.dual() + (stack((mv.x.dual(), mv.y.dual(), mv.z.dual())) * xyz).sum(axis=-1)


assert len(states) == n_frames
assert ((generator.squared() + 1).scalar_norm_squared() < 1e-20).all()
last_quadrics = states[-1]                                                     # [3] Plane+ <- Point-
probes = sphere_points(np.array([[0.6, 0.0, 0.8], [0.0, -0.28, 0.96], [0.36, 0.48, 0.8]]))
assert (((last_quadrics.adjugate()(probes[0, None]) & probes[1]) - (last_quadrics(probes[1, None]) & probes[0])).abs() < 1e-12).all()

one_step = (generator * (np.pi / n_frames)).exp()                              # [] Rotor+
closed = one_step >> last_quadrics(one_step << Point)                          # [3] Plane+ <- Point-
assert (((closed(probes[:, None]) & probes[:, None]) - (quadrics(probes[:, None]) & probes[:, None])).abs() < 1e-6).all()
carried_probes = one_step >> probes                                            # [3] Point-
assert ((carried_probes | carried_probes).abs() < 1e-12).all()